This script simulates spacecraft telemetry data, integrating information regarding temperature, pressure, energy levels, and the status of critical systems.
The telemetry is organized in a tabular format (DataFrame) and saved as a CSV file for subsequent computational analysis.

In [12]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

ROWS = 1000
START_TIME = datetime(2026, 3, 14, 10, 0, 0)
OUTPUT_PATH = "telemetry_dataset.csv"

def generate_telemetry(rows, start_time):
    timestamps = [start_time + timedelta(seconds=i) for i in range(rows)]

    temp_internal = np.clip(
        34 + np.cumsum(np.random.normal(0.05, 0.2, rows)),
        30, 85
    )

    temp_external = np.clip(
        18 + np.random.normal(0, 1.2, rows),
        -20, 40
    )

    energy_level = np.clip(
        100 - np.linspace(0, 40, rows) + np.random.normal(0, 0.5, rows),
        0, 100
    )

    tank_pressure = np.clip(
        250 - np.linspace(0, 40, rows) + np.random.normal(0, 0.8, rows),
        150, 260
    )

    # Sistemas binários
    structural_integrity = np.ones(rows, dtype=int)
    structural_integrity[np.random.choice(rows, size=5, replace=False)] = 0

    def generate_status():
        arr = np.ones(rows, dtype=int)
        arr[np.random.choice(rows, size=3, replace=False)] = 0
        return arr

    df = pd.DataFrame({
        "timestamp": timestamps,
        "temp_internal_c": temp_internal.round(2),
        "temp_external_c": temp_external.round(2),
        "structural_integrity": structural_integrity,
        "energy_level_pct": energy_level.round(2),
        "tank_pressure_bar": tank_pressure.round(2),
        "avionics_status": generate_status(),
        "propulsion_status": generate_status(),
        "navigation_status": generate_status(),
        "comms_status": generate_status()
    })

    return df

df = generate_telemetry(ROWS, START_TIME)
df.head()

df.to_csv(OUTPUT_PATH, index=False)
print(f"Dataset salvo em: {OUTPUT_PATH}")


Dataset salvo em: telemetry_dataset.csv


The script aims to load a flight telemetry dataset (telemetry_dataset.csv) and perform an initial visualization of the first entries, using Python and data analysis and visualization libraries.

In [13]:
# Importações
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (14, 6)

# Carregar dataset
df = pd.read_csv("telemetry_dataset.csv", parse_dates=["timestamp"])
df.head()

,timestamp,temp_internal_c,temp_external_c,structural_integrity,energy_level_pct,tank_pressure_bar,avionics_status,propulsion_status,navigation_status,comms_status
0,2026-03-14 10:00:00,33.66,15.59,1,99.24,249.77,1,1,1,1
1,2026-03-14 10:00:01,33.93,15.67,1,99.99,249.85,1,1,1,1
2,2026-03-14 10:00:02,34.24,18.27,1,99.74,249.15,1,1,1,1
3,2026-03-14 10:00:03,34.36,16.32,1,99.79,249.84,1,1,1,1
4,2026-03-14 10:00:04,34.38,16.80,1,99.35,250.45,1,1,1,1


In [14]:
# Statistics Describe

df.describe()

,timestamp,temp_internal_c,temp_external_c,structural_integrity,energy_level_pct,tank_pressure_bar,avionics_status,propulsion_status,navigation_status,comms_status
count,1000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,2026-03-14 10:08:19.500000,59.017830,17.939170,0.995000,79.980870,230.033290,0.997000,0.997000,0.997000,0.997000
min,2026-03-14 10:00:00,33.660000,14.680000,0.000000,59.070000,208.940000,0.000000,0.000000,0.000000,0.000000
25%,2026-03-14 10:04:09.750000,47.967500,17.140000,1.000000,70.012500,220.252500,1.000000,1.000000,1.000000,1.000000
50%,2026-03-14 10:08:19.500000,59.995000,17.980000,1.000000,79.895000,230.055000,1.000000,1.000000,1.000000,1.000000
75%,2026-03-14 10:12:29.250000,69.802500,18.710000,1.000000,89.865000,240.112500,1.000000,1.000000,1.000000,1.000000
max,2026-03-14 10:16:39,82.820000,21.660000,1.000000,100.000000,250.970000,1.000000,1.000000,1.000000,1.000000
std,NaN,13.256274,1.192848,0.070569,11.593013,11.587709,0.054717,0.054717,0.054717,0.054717


In [15]:
# Mission Confi

TOTAL_CAPACITY_KWH = 500
TAKEOFF_POWER_KW = 120
ENERGY_LOSS = 0.08
INITIAL_PAYLOAD_KG = 800

Energy Metrics (calculate_energy_metrics)

This function calculates the key indicators related to the energy available and consumed during the mission.

Outputs: A dictionary containing:

initial_energy_pct: initial energy percentage
energy_available_kwh: available energy (kWh)
energy_usable_kwh: usable energy (kWh)
initial_autonomy_h: initial autonomy (h)
adjusted_energy_usable_kwh: adjusted energy (considering thermal losses)
adjusted_autonomy_h: adjusted autonomy (h)
thermal_loss_factor: thermal loss factor

Theoretical Reference: Energy efficiency in embedded systems, thermal loss management, and energy consumption optimization

In [16]:
# =========================================
# 4. ENERGY METRICS
# =========================================
def calculate_energy_metrics(df):

    initial_energy_pct = float(df["energy_level_pct"].iloc[0])

    energy_available = TOTAL_CAPACITY_KWH * (initial_energy_pct / 100.0)
    energy_usable = energy_available * (1.0 - ENERGY_LOSS)

    initial_autonomy = energy_usable / TAKEOFF_POWER_KW

    temp_internal_mean = float(df["temp_internal_c"].mean())
    thermal_loss_factor = 1.0 + ((temp_internal_mean - 30.0) / 100.0)

    adjusted_energy_usable = energy_usable / thermal_loss_factor
    adjusted_autonomy = adjusted_energy_usable / TAKEOFF_POWER_KW

    return {
        "initial_energy_pct": initial_energy_pct,
        "energy_available_kwh": energy_available,
        "energy_usable_kwh": energy_usable,
        "initial_autonomy_h": initial_autonomy,
        "adjusted_energy_usable_kwh": adjusted_energy_usable,
        "adjusted_autonomy_h": adjusted_autonomy,
        "thermal_loss_factor": thermal_loss_factor
    }

System Metrics (calculate_system_metrics)

Calculates the overall condition of the ship's critical systems.

Outputs: A dictionary containing:

avg_pressure_bar: average tank pressure
fuel_consumption_index: consumption index
structural_ok: structural integrity
systems_ok: critical systems status
mission_status: final result of the mission check

Theoretical Reference: The importance of structural integrity and critical systems for a safe launch.

In [17]:
# =========================================
# 5. SYSTEM METRICS
# =========================================
def calculate_system_metrics(df):

    avg_pressure = float(df["tank_pressure_bar"].mean())

    fuel_consumption_index = TAKEOFF_POWER_KW * (1.0 + (INITIAL_PAYLOAD_KG / 1000.0))

    structural_ok = bool(df["structural_integrity"].all())

    systems_ok = all([
        df["avionics_status"].all(),
        df["propulsion_status"].all(),
        df["navigation_status"].all(),
        df["comms_status"].all()
    ])

    mission_status = "🚀 READY FOR TAKEOFF" if structural_ok and systems_ok else "❌ LAUNCH ABORTED"

    return {
        "avg_pressure_bar": avg_pressure,
        "fuel_consumption_index": fuel_consumption_index,
        "structural_ok": structural_ok,
        "systems_ok": systems_ok,
        "mission_status": mission_status
    }


Loss Metrics (calculate_loss_metrics)

Evaluates energy losses and auxiliary loads.

Outputs: DataFrame with the following columns:

thermal_loss
mechanical_loss
auxiliary_load
energy_level_pct

Theoretical Reference: Energy loss management, consumption optimization in critical systems

Correlations (calculate_correlations)

Calculates the correlation matrix among loss metrics to identify significant relationships.

Outputs: Correlation matrix (DataFrame)

Theoretical Reference: Correlation analysis to detect dependencies between physical and computational variables

This set of functions enables:

Evaluating the spacecraft's energy autonomy, taking thermal conditions into account;
Monitoring structural and operational integrity;
Quantifying energy losses and auxiliary system loads;
Establishing relationships between critical variables for performance and risk analyses.

In [18]:
# =========================================
# 6. LOSS METRICS
# =========================================
def calculate_loss_metrics(df):

    loss_df = pd.DataFrame(index=df.index)

    loss_df["thermal_loss"] = (df["temp_internal_c"] - 30).clip(lower=0)

    loss_df["mechanical_loss"] = abs(
        df["tank_pressure_bar"] - df["tank_pressure_bar"].mean()
    )

    loss_df["auxiliary_load"] = (
        df["avionics_status"] +
        df["navigation_status"] +
        df["comms_status"]
    )

    loss_df["energy_level_pct"] = df["energy_level_pct"]

    return loss_df

# =========================================
# 7. CORRELATION
# =========================================
def calculate_correlations(loss_df):
    return loss_df.corr(numeric_only=True)

In [ ]:
# =========================================
# 8. EXECUTION
# =========================================
energy_metrics = calculate_energy_metrics(df)
system_metrics = calculate_system_metrics(df)

loss_df = calculate_loss_metrics(df)
corr_matrix = calculate_correlations(loss_df)

print("Energy Metrics",energy_metrics)
print("System Metrics",system_metrics)
print("Loss",loss_df)
print("Correlation Loss",corr_matrix)


